# [INFO-H515 - Big Data Scalable Analytics](https://uv.ulb.ac.be/course/view.php?id=85246?username=guest)

## TP 5 - Examn question : Matrix transposition

#### *Gianluca Bontempi, Cédric Simar, and Martin Colot - materials from Jacopo De Stefani and Theo Verhelst*

####  2025


## General imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession

## Data generation

In [ ]:
def genData(N, n, random_seed):
    np.random.seed(random_seed)

    #Inputs and the weights of the linear combination are drawn at random
    X = np.random.rand(N, n)
    theta = np.random.rand(n)
    noise = np.random.rand(N)

    Y = np.dot(X, theta) #+ noise
    Y = Y[:, np.newaxis]
    Z = np.concatenate((Y, X), axis=1)

    print("Number of observations :", N)
    print("Number of features :", n)

    print("Dimension of X :", X.shape)
    print("Dimension of theta :", theta.shape)
    print("Dimension of Y :", Y.shape)

    return (X, Y, Z, theta)

In [ ]:
X, Y, Z, theta = genData(1000, 10, 50)

Number of observations : 1000
Number of features : 10
Dimension of X : (1000, 10)
Dimension of theta : (10,)
Dimension of Y : (1000, 1)


In [ ]:
data = Z.transpose()
data = np.concatenate((np.arange(len(data)).reshape(1, -1).transpose(), data), axis=1)

In [ ]:
data.shape

(11, 1001)

In [ ]:
pd.DataFrame(data).head()

,0,1,2,3,4,5,6,7,8,9,...,991,992,993,994,995,996,997,998,999,1000
0,0.0,2.486938,2.205703,2.582589,2.415691,1.803551,1.759056,1.447913,1.602298,1.788159,...,2.177410,1.658852,2.759405,1.285563,2.083369,0.758870,2.314843,2.395534,2.810104,2.303804
1,1.0,0.548814,0.791725,0.978618,0.264556,0.359508,0.570197,0.158970,0.976459,0.317983,...,0.884982,0.281426,0.589455,0.246941,0.039411,0.166566,0.911452,0.409317,0.478317,0.694983
2,2.0,0.715189,0.528895,0.799159,0.774234,0.437032,0.438602,0.110375,0.468651,0.414263,...,0.197014,0.186771,0.919616,0.517327,0.857732,0.263630,0.057940,0.816221,0.846360,0.194607
3,3.0,0.602763,0.568045,0.461479,0.456150,0.697631,0.988374,0.656330,0.976761,0.064147,...,0.568613,0.654647,0.223506,0.572553,0.891708,0.192304,0.671770,0.011549,0.548319,0.114037
4,4.0,0.544883,0.925597,0.780529,0.568434,0.060225,0.102045,0.138183,0.604846,0.692472,...,0.931015,0.229331,0.453969,0.552292,0.122564,0.832469,0.349697,0.787698,0.991371,0.262116


## Spark Session

In [ ]:
# Initialize Spark Session (local mode)
# Start Spark session with local master and 2 cores
spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("LinearRegression") \
    .getOrCreate()

# Let us retrieve the sparkContext object
sc = spark.sparkContext

spark.conf.set("spark.sql.shuffle.partitions", "2") # reduce number of partitions to run on google Colab

In [ ]:
#Let us use 2 partitions
B = 2
Z_RDD = sc.parallelize(data, B).cache()

In [ ]:
np.array(Z_RDD.collect()).shape

(11, 1001)

**Exercise:**

Let us consider a least-squares regression problem with a single output target Y and n input variables $X_1, \dots, X_n$. Suppose that the dataset (made of $N$ observations) is stored in a horizontal format, i.e. the first row corresponds to the output Y and the following n rows correspond to the $n$ inputs. Also, the first column contains the ID of the line.

\begin{equation}
\begin{pmatrix}
  0       & Y_0   & Y_1   & \cdots  & Y_n  \\
  1       & X_{1,1}   & X_{1,2}   & \cdots  & X_{1,N}  \\
  \vdots  & \vdots  & \vdots  & \ddots  & \vdots \\
  n       & X_{n,1}   & X_{n,2}   & \cdots  & X_{n,N}  \\
\end{pmatrix}
\end{equation}


Use map/reduce operations to transpose the table (stored in Z_RDD) and remove the IDs of the features and label (first column of Z_RDD). The resulting table should have a shape of (1000, 11), where the first column contains the labels, and the remaining columns contain the features in the correct order

**Your Solution**

**Solution**

1. use flat map to create one line per value in the table, where the first column corresponds to the sample number. Each line contains a list of one tuple with the feature/label ID, and the value

In [ ]:
a = Z_RDD.flatMap(lambda x : [(i, [(x[0], x[i])]) for i in range(len(x))])
pd.DataFrame(a.collect())

,0,1
0,0,"[(0.0, 0.0)]"
1,1,"[(0.0, 2.4869377458373854)]"
2,2,"[(0.0, 2.2057028064625053)]"
3,3,"[(0.0, 2.582589248991649)]"
4,4,"[(0.0, 2.41569055452804)]"
...,...,...
11006,996,"[(10.0, 0.03169086549665512)]"
11007,997,"[(10.0, 0.8712535814827824)]"
11008,998,"[(10.0, 0.8484203920653359)]"
11009,999,"[(10.0, 0.9910432025957882)]"


2. Groups the lines based on the sample number and use list concatenation to obtain one list of tuples (feature/label id, value) per sample

In [ ]:
b = a.reduceByKey(lambda x, y : x + y)
pd.DataFrame(b.collect())

,0,1
0,0,"[(0.0, 0.0), (1.0, 1.0), (2.0, 2.0), (3.0, 3.0..."
1,2,"[(0.0, 2.2057028064625053), (1.0, 0.7917250380..."
2,4,"[(0.0, 2.41569055452804), (1.0, 0.264555612104..."
3,6,"[(0.0, 1.7590556970293885), (1.0, 0.5701967704..."
4,8,"[(0.0, 1.602298372179235), (1.0, 0.97645946501..."
...,...,...
996,991,"[(0.0, 2.1774101613520256), (1.0, 0.8849823177..."
997,993,"[(0.0, 2.7594049116730277), (1.0, 0.5894545835..."
998,995,"[(0.0, 2.0833690457743073), (1.0, 0.0394105054..."
999,997,"[(0.0, 2.3148425294657398), (1.0, 0.9114515269..."


3. For each sample, sort the features/labels based on their ID, and remove these IDs

In [ ]:
c = b.map(lambda x : (x[0], sorted(x[1], key=lambda e: e[0]))).map(lambda x : (x[0], [e[1] for e in x[1]]))
pd.DataFrame(c.collect())

,0,1
0,0,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
1,2,"[2.2057028064625053, 0.7917250380826646, 0.528..."
2,4,"[2.41569055452804, 0.26455561210462697, 0.7742..."
3,6,"[1.7590556970293885, 0.5701967704178796, 0.438..."
4,8,"[1.602298372179235, 0.9764594650133958, 0.4686..."
...,...,...
996,991,"[2.1774101613520256, 0.8849823177063212, 0.197..."
997,993,"[2.7594049116730277, 0.5894545835657162, 0.919..."
998,995,"[2.0833690457743073, 0.0394105054556555, 0.857..."
999,997,"[2.3148425294657398, 0.9114515269999425, 0.057..."


4. Remove the sample with sample number equal to $0$ as it contains the features/label IDs

In [ ]:
d = c.filter(lambda x : x[0] > 0).map(lambda x : x[1])
pd.DataFrame(d.collect())

,0,1,2,3,4,5,6,7,8,9,10
0,2.205703,0.791725,0.528895,0.568045,0.925597,0.071036,0.087129,0.020218,0.832620,0.778157,0.870012
1,2.415691,0.264556,0.774234,0.456150,0.568434,0.018790,0.617635,0.612096,0.616934,0.943748,0.681820
2,1.759056,0.570197,0.438602,0.988374,0.102045,0.208877,0.161310,0.653108,0.253292,0.466311,0.244426
3,1.602298,0.976459,0.468651,0.976761,0.604846,0.739264,0.039188,0.282807,0.120197,0.296140,0.118728
4,1.082989,0.318569,0.667410,0.131798,0.716327,0.289406,0.183191,0.586513,0.020108,0.828940,0.004695
...,...,...,...,...,...,...,...,...,...,...,...
995,2.177410,0.884982,0.197014,0.568613,0.931015,0.564493,0.211629,0.264954,0.678571,0.747040,0.591752
996,2.759405,0.589455,0.919616,0.223506,0.453969,0.965818,0.954882,0.511586,0.448669,0.944827,0.599457
997,2.083369,0.039411,0.857732,0.891708,0.122564,0.461569,0.393185,0.126154,0.864393,0.864067,0.740818
998,2.314843,0.911452,0.057940,0.671770,0.349697,0.455498,0.221139,0.338526,0.308106,0.708900,0.871254
